# 05 — Exportação para o firmware de inferência

Gera, a partir dos modelos dos notebooks 03 v2 e 04, todos os arquivos `.h` que o firmware precisa:

| Arquivo | Conteúdo |
|---|---|
| `modelo_cnn.h` | CNN 2D int8 (camadas densas quantizadas **por tensor**; ver célula 2) |
| `modelo_lstm.h` | LSTM de passo único em **float32** (precisão mista, seção 8.2) |
| `ops_modelos.h` | Lista exata de operações usadas pelos dois modelos (para o `MicroMutableOpResolver`) |
| `parametros_modelos.h` | Escalas e zeros da CNN, ordem das entradas/saídas do LSTM, faixas min-max, arenas mínimas |
| `tabelas_mel.h` | Janela de Hann, banco mel esparso e tabelas da FFT de 512 pontos |
| `vetores_teste.h` | Áudio de paridade + saídas esperadas (espectrograma, CNN, LSTM) — **só para o sketch de teste** |

Também valida os dois modelos no **TensorFlow Lite Micro** (o mesmo interpretador da placa, versão atual) e mede a arena mínima.

**Ambiente:** o mesmo dos notebooks 03 v2 e 04 (GPU T4) ou CPU. Se o ambiente não tiver TensorFlow, a célula 1 instala a versão 2.21.0. Leva poucos minutos (a avaliação da CNN no teste é a parte mais lenta).

In [ ]:
# 1. Ambiente, caminhos e interpretadores
import os, sys, re, json, zipfile, subprocess, importlib.util
# Algumas imagens do Colab (CPU) vêm sem TensorFlow: instala a MESMA versão usada no treino (2.21.0),
# para ler os .keras e converter exatamente como nos notebooks 03 v2 e 04.
if importlib.util.find_spec("tensorflow") is None:
    print("Instalando TensorFlow 2.21.0 (1 a 2 min)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow==2.21.0"], check=True)
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.metrics import accuracy_score, f1_score
print("Python", sys.version.split()[0], "| TensorFlow", tf.__version__, "| NumPy", np.__version__)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/dissertacao_tinyml"
except ImportError:
    BASE = os.environ.get("BASE_TINYML", "dissertacao_tinyml")
VERSAO_CNN = "cnn_v3"            # cnn_v3: passo 2 na 1ª convolução (embarcada); cnn_v2: versão anterior
MIMII, CNN, LSTM, BRISTOL = (f"{BASE}/{p}" for p in ["mimii_fan_6db", f"modelos/{VERSAO_CNN}", "modelos/lstm", "bristol_preparado"])
print("CNN:", VERSAO_CNN)
SAIDA = f"{BASE}/firmware"
os.makedirs(SAIDA, exist_ok=True)

try:
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    Interpreter = tf.lite.Interpreter
try:
    from tflite_micro.python.tflite_micro import runtime as tflm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tflite-micro"], check=False)
    try:
        from tflite_micro.python.tflite_micro import runtime as tflm
    except ImportError:
        tflm = None
print("TFLite Micro no host:", "disponível" if tflm else "indisponível (a célula 4 será pulada)")

In [ ]:
# 2. CNN: nova conversão int8 com as camadas densas quantizadas POR TENSOR
# O TF 2.21 quantiza as camadas densas por canal. O TFLite Micro da Harvard_TinyMLx (base TF 2.4) só aceita
# FULLY_CONNECTED por tensor: com pesos por canal, a placa calcularia a saída errada sem avisar.
# As convoluções continuam por canal (suportadas). Mesmo modelo, mesmas 100 amostras de calibração do notebook 03 v2.
qjs = json.load(open(f"{CNN}/cnn_int8_quantizacao.json"))
SEMENTE_CNN = qjs["semente"]
modelo_cnn = tf.keras.models.load_model(f"{CNN}/cnn_semente{SEMENTE_CNN}.keras")
STATS = {m: (np.array(v["media"], np.float32), np.array(v["desvio"], np.float32))
         for m, v in qjs["linhas_de_base_mimii"].items()}
MAQ = pd.read_csv(f"{MIMII}/particao_por_gravacao.csv")["maquina"].values

def carregar(p):
    d = np.load(f"{MIMII}/{p}.npz")
    return d["X"], d["y"].astype(np.int32), d["grupo"]

def normalizar(X, g):
    out = np.empty(X.shape, np.float32)
    for m, (mu, sd) in STATS.items():
        s = MAQ[g] == m
        out[s] = (X[s].astype(np.float32) - mu) / sd
    return out[..., None]

Xtr, ytr, gtr = carregar("treino")
rng = np.random.default_rng(0)
idx_rep = rng.choice(len(Xtr), 100, replace=False)          # mesmas 100 amostras do notebook 03 v2
Xrep = normalizar(Xtr[idx_rep], gtr[idx_rep])
del Xtr, ytr, gtr

def converter_cnn(por_tensor):
    c = tf.lite.TFLiteConverter.from_keras_model(modelo_cnn)
    c.optimizations = [tf.lite.Optimize.DEFAULT]
    c.representative_dataset = lambda: ([Xrep[i:i+1]] for i in range(len(Xrep)))
    c.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    c.inference_input_type = tf.int8
    c.inference_output_type = tf.int8
    if por_tensor:
        c._experimental_disable_per_channel_quantization_for_dense_layers = True
    return c.convert()

cnn_canal = open(f"{CNN}/cnn_int8.tflite", "rb").read()       # versão das Tabelas 10 e 11
cnn_f32 = open(f"{CNN}/cnn_float32.tflite", "rb").read()
cnn_tensor = converter_cnn(True)
open(f"{CNN}/cnn_int8_densa_por_tensor.tflite", "wb").write(cnn_tensor)

def quant(d): return [float(v) for v in d["quantization"]]
def detalhes(b):
    it = Interpreter(model_content=b); it.allocate_tensors()
    return it, it.get_input_details(), it.get_output_details()

def por_canal(b):
    it, _, _ = detalhes(b)
    return [t["name"] for t in it.get_tensor_details()
            if len(t["quantization_parameters"]["scales"]) > 1 and "MatMul" in t["name"]]
print("Densas por canal — versão original:", por_canal(cnn_canal))
print("Densas por canal — nova versão:   ", por_canal(cnn_tensor) or "nenhuma")

def prever_cnn(b, X):
    it, ent, sai = detalhes(b); ent, sai = ent[0], sai[0]
    out = np.zeros((len(X), sai["shape"][-1]), np.float32)
    for i in range(len(X)):
        x = X[i:i+1]
        if ent["dtype"] == np.int8:
            s, z = ent["quantization"]; x = np.clip(np.round(x / s + z), -128, 127).astype(np.int8)
        it.set_tensor(ent["index"], x); it.invoke()
        y = it.get_tensor(sai["index"]).astype(np.float32)
        out[i] = (y - sai["quantization"][1]) * sai["quantization"][0] if sai["dtype"] == np.int8 else y
    return out

Xte, yte, gte = carregar("teste"); Xte = normalizar(Xte, gte)
P = {n: prever_cnn(b, Xte) for n, b in [("float32", cnn_f32), ("int8 (notebook 03)", cnn_canal), ("int8 densa por tensor", cnn_tensor)]}
comp = pd.DataFrame({n: {"acurácia": accuracy_score(yte, p.argmax(1)),
                         "F1 macro": f1_score(yte, p.argmax(1), average="macro"),
                         "concordância c/ float32 (%)": 100 * (p.argmax(1) == P["float32"].argmax(1)).mean(),
                         "tamanho (KB)": len(b) / 1024}
                     for (n, p), b in zip(P.items(), [cnn_f32, cnn_canal, cnn_tensor])})
try:
    display(comp.round(4))
except NameError:
    print(comp.round(4))
comp.to_csv(f"{CNN}/float32_vs_int8_densa_por_tensor.csv")
_, ent, sai = detalhes(cnn_tensor)
CNN_Q_ENT, CNN_Q_SAI = quant(ent[0]), quant(sai[0])
print("Entrada (escala, zero):", CNN_Q_ENT, "| saída:", CNN_Q_SAI)

In [ ]:
# 3. LSTM de passo único (float32): ordem das entradas e saídas + verificação contra o modelo Keras completo
FAIXAS = json.load(open(f"{BRISTOL}/params_bristol.json"))["faixas_minmax"]
FX = [FAIXAS["temp"], FAIXAS["umid"], FAIXAS["dpress"]]
def atributos(v):                                    # idêntico ao notebook 02 e ao firmware
    v = np.asarray(v, np.float32)
    dp = v[..., 2] - v[..., 2].mean(axis=-1, keepdims=True)
    cols = [v[..., 0], v[..., 1], dp]
    return np.stack([np.clip((c - lo) / (hi - lo), 0, 1) for c, (lo, hi) in zip(cols, FX)], axis=-1).astype(np.float32)

lstm_b = open(f"{LSTM}/lstm_passo_float32.tflite", "rb").read()
it_l, ent_l, sai_l = detalhes(lstm_b)
PAPEL_ENT = ["x" if "x_t" in d["name"] else "h" if d["name"].endswith("_h:0") else "c" for d in ent_l]
ordem = sorted(range(len(sai_l)), key=lambda i: int(sai_l[i]["name"].split(":")[-1]))   # nomes :0, :1, :2 = h', c', prob
PAPEL_SAI = [None] * 3
for papel, i in zip(["h", "c", "p"], ordem):
    PAPEL_SAI[i] = papel
assert sorted(PAPEL_ENT) == ["c", "h", "x"] and sai_l[PAPEL_SAI.index("p")]["shape"][-1] == 4
print("Entradas (na ordem do modelo):", [(d["name"], list(d["shape"])) for d in ent_l], "->", PAPEL_ENT)
print("Saídas   (na ordem do modelo):", [(d["name"], list(d["shape"])) for d in sai_l], "->", PAPEL_SAI)

def rodar_lstm(b, Xn):
    it, ent, sai = detalhes(b)
    E = {PAPEL_ENT[i]: d["index"] for i, d in enumerate(ent)}; S = {PAPEL_SAI[i]: d["index"] for i, d in enumerate(sai)}
    U = ent[PAPEL_ENT.index("h")]["shape"][-1]; out = np.zeros((len(Xn), 4), np.float32)
    for n in range(len(Xn)):
        h = np.zeros((1, U), np.float32); c = np.zeros((1, U), np.float32)
        for t in range(Xn.shape[1]):
            it.set_tensor(E["x"], Xn[n, t:t+1]); it.set_tensor(E["h"], h); it.set_tensor(E["c"], c); it.invoke()
            h, c = it.get_tensor(S["h"]).copy(), it.get_tensor(S["c"]).copy()
        out[n] = it.get_tensor(S["p"])[0]
    return out

ljs = json.load(open(f"{LSTM}/lstm_int8_quantizacao.json"))
SEMENTE_LSTM = ljs["semente"]
modelo_lstm = tf.keras.models.load_model(f"{LSTM}/lstm_semente{SEMENTE_LSTM}.keras")
bt = np.load(f"{BRISTOL}/teste.npz")
print("Atributos do firmware = atributos do treino:", float(np.abs(atributos(bt["X_bruto"][:500]) - bt["X"][:500]).max()))
amostra = bt["X"][:50]
dif = np.abs(rodar_lstm(lstm_b, amostra) - modelo_lstm.predict(amostra, verbose=0)).max()
print(f"Passo único (TFLite, 30 passos) x LSTM Keras completo: diferença máxima {dif:.2e}")
assert dif < 1e-4, "ordem de entradas/saídas inconsistente"

# Pesos para a execução do LSTM em C, sem o interpretador (mesmos pesos, mesma ordem de portas i, f, g, o do Keras)
Wk, Wr, bl = modelo_lstm.get_layer("lstm").get_weights()
Wd, bd = modelo_lstm.get_layer("densa").get_weights()
Wo, bo = modelo_lstm.get_layer("saida").get_weights()
def lstm_c(Xn):                                      # mesma conta do lstm_c.h, em float32
    sig = lambda v: 1 / (1 + np.exp(-v))
    out = np.zeros((len(Xn), 4), np.float32)
    for n in range(len(Xn)):
        h = np.zeros(32, np.float32); c = np.zeros(32, np.float32)
        for t in range(Xn.shape[1]):
            z = Xn[n, t] @ Wk + h @ Wr + bl
            i, f, g, o = sig(z[:32]), sig(z[32:64]), np.tanh(z[64:96]), sig(z[96:])
            c = f * c + i * g; h = o * np.tanh(c)
        d = np.maximum(h @ Wd + bd, 0); e = np.exp(d @ Wo + bo - (d @ Wo + bo).max()); out[n] = e / e.sum()
    return out
dif_c = np.abs(lstm_c(amostra) - rodar_lstm(lstm_b, amostra)).max()
print(f"LSTM em C (conta em NumPy) x TFLite: diferença máxima {dif_c:.2e}")
assert dif_c < 1e-4


In [ ]:
# 4. Validação no TensorFlow Lite Micro e arena mínima (busca binária)
# Durante a busca, o TFLM imprime mensagens "Failed to allocate..." para as arenas pequenas demais: é esperado.
def ops_do_modelo(b):
    it = Interpreter(model_content=b); it.allocate_tensors()
    return sorted({d["op_name"] for d in it._get_ops_details()} - {"DELEGATE"})
OPS = sorted(set(ops_do_modelo(cnn_tensor)) | set(ops_do_modelo(lstm_b)))
print("CNN :", ops_do_modelo(cnn_tensor)); print("LSTM:", ops_do_modelo(lstm_b))

ARENA = {"cnn": None, "lstm": None}
if tflm:
    def tflm_roda(b, n_ent, arena):
        try:
            it = tflm.Interpreter.from_bytes(b, arena_size=arena)
            for i in range(n_ent):
                d = it.get_input_details(i); it.set_input(np.zeros(d["shape"], d["dtype"]), i)
            it.invoke(); return it
        except Exception:
            return None
    def arena_minima(b, n_ent):
        lo, hi = 512, 512 * 1024
        while hi - lo > 64:
            mid = (lo + hi) // 2
            lo, hi = (lo, mid) if tflm_roda(b, n_ent, mid) else (mid, hi)
        return hi
    ARENA = {"cnn": arena_minima(cnn_tensor, 1), "lstm": arena_minima(lstm_b, 3)}
    print(f"Arena mínima no TFLM atual: CNN {ARENA['cnn']/1024:.1f} KB | LSTM {ARENA['lstm']/1024:.1f} KB")

    # mesmas saídas que o TFLite? CNN (int8, 20 janelas) e LSTM (float32, 5 janelas)
    it_c, ent_c, sai_c = detalhes(cnn_tensor); s, z = ent_c[0]["quantization"]
    dif_c = 0
    for i in range(20):
        q = np.clip(np.round(Xte[i:i+1] / s + z), -128, 127).astype(np.int8)
        it_c.set_tensor(ent_c[0]["index"], q); it_c.invoke(); ref = it_c.get_tensor(sai_c[0]["index"]).astype(int)
        m = tflm.Interpreter.from_bytes(cnn_tensor, arena_size=ARENA["cnn"] + 4096); m.set_input(q, 0); m.invoke()
        dif_c = max(dif_c, int(np.abs(m.get_output(0).astype(int) - ref).max()))
    print("CNN: maior diferença TFLM x TFLite (em passos de quantização):", dif_c)
    m = tflm.Interpreter.from_bytes(lstm_b, arena_size=ARENA["lstm"] + 4096)
    E = {PAPEL_ENT[i]: i for i in range(3)}; S = {PAPEL_SAI[i]: i for i in range(3)}
    dif_l = 0; ref = rodar_lstm(lstm_b, amostra[:5])
    for n in range(5):
        h = np.zeros((1, 32), np.float32); c = np.zeros((1, 32), np.float32)
        for t in range(30):
            m.set_input(amostra[n, t:t+1], E["x"]); m.set_input(h, E["h"]); m.set_input(c, E["c"]); m.invoke()
            h, c = m.get_output(S["h"]).copy(), m.get_output(S["c"]).copy()
        dif_l = max(dif_l, float(np.abs(m.get_output(S["p"])[0] - ref[n]).max()))
    print(f"LSTM: maior diferença TFLM x TFLite: {dif_l:.2e}")

In [ ]:
# 5. Geração dos arquivos .h
def num(v, tipo):
    if tipo == "float":
        t = f"{float(v):.9g}"
        return (t if any(ch in t for ch in ".en") else t + ".0") + "f"
    return str(int(v))

def vetor(tipo, nome, dados, por_linha=12, alinhado=False):
    dados = np.asarray(dados).ravel()
    linhas = [", ".join(num(v, tipo) for v in dados[i:i+por_linha]) for i in range(0, len(dados), por_linha)]
    pre = "alignas(16) " if alinhado else ""
    return f"{pre}const {tipo} {nome}[{len(dados)}] = {{\n  " + ",\n  ".join(linhas) + "\n};\n"

def bytes_c(nome, b):
    return ("#pragma once\n// Gerado por 05_exportar_firmware.ipynb — não editar\n#include <stdint.h>\n"
            + vetor("unsigned char", nome, np.frombuffer(b, np.uint8), por_linha=16, alinhado=True)
            + f"const unsigned int {nome}_len = {len(b)};\n")

CAB = "#pragma once\n// Gerado por 05_exportar_firmware.ipynb — não editar\n#include <stdint.h>\n\n"
arquivos = {}
arquivos["modelo_cnn.h"] = bytes_c("g_modelo_cnn", cnn_tensor)
arquivos["modelo_lstm.h"] = bytes_c("g_modelo_lstm", lstm_b)
arquivos["pesos_lstm.h"] = (CAB + f"// Pesos do LSTM (semente {SEMENTE_LSTM}) para a execução em C (lstm_c.h). Matrizes [entrada][saída].\n"
    + "// Portas na ordem do Keras: i, f, g (candidato), o.\n"
    + vetor("float", "LSTM_WK", Wk, 8) + vetor("float", "LSTM_WR", Wr, 8) + vetor("float", "LSTM_B", bl, 8)
    + vetor("float", "LSTM_WD", Wd, 8) + vetor("float", "LSTM_BD", bd, 8)
    + vetor("float", "LSTM_WO", Wo, 8) + vetor("float", "LSTM_BO", bo, 4))

METODO = {"CONV_2D": "AddConv2D", "MAX_POOL_2D": "AddMaxPool2D", "AVERAGE_POOL_2D": "AddAveragePool2D",
          "MEAN": "AddMean", "FULLY_CONNECTED": "AddFullyConnected", "SOFTMAX": "AddSoftmax", "ADD": "AddAdd",
          "SUB": "AddSub", "MUL": "AddMul", "LOGISTIC": "AddLogistic", "TANH": "AddTanh", "RESHAPE": "AddReshape",
          "STRIDED_SLICE": "AddStridedSlice", "SPLIT": "AddSplit", "CONCATENATION": "AddConcatenation",
          "QUANTIZE": "AddQuantize", "DEQUANTIZE": "AddDequantize", "PACK": "AddPack", "UNPACK": "AddUnpack"}
faltam = [o for o in OPS if o not in METODO]
assert not faltam, f"operações sem mapeamento: {faltam}"
arquivos["ops_modelos.h"] = (CAB + f"// Operações usadas pela CNN e pelo LSTM: {', '.join(OPS)}\n"
    f"#define N_OPS_MODELOS {len(OPS)}\n\ntemplate <typename R>\nbool registrarOps(R& r) {{\n  return\n"
    + " &&\n".join(f"    r.{METODO[o]}() == kTfLiteOk" for o in OPS) + ";\n}\n")

ie = {p: i for i, p in enumerate(PAPEL_ENT)}; isai = {p: i for i, p in enumerate(PAPEL_SAI)}
arena_cnn = ARENA["cnn"] or 48 * 1024; arena_lstm = ARENA["lstm"] or 8 * 1024
arquivos["parametros_modelos.h"] = CAB + f'''// CNN 2D (int8; semente {SEMENTE_CNN}): entrada 98 x 40 x 1, saída [normal, anômalo]
const int   CNN_QUADROS = 98, CNN_FILTROS_MEL = 40;
const float CNN_ESCALA_ENTRADA = {num(CNN_Q_ENT[0], "float")};
const int   CNN_ZERO_ENTRADA   = {int(CNN_Q_ENT[1])};
const float CNN_ESCALA_SAIDA   = {num(CNN_Q_SAI[0], "float")};
const int   CNN_ZERO_SAIDA     = {int(CNN_Q_SAI[1])};

// LSTM de passo único (float32; semente {SEMENTE_LSTM}): índices na ordem do modelo
const int LSTM_PASSOS = 30, LSTM_ATRIBUTOS = 3, LSTM_UNIDADES = 32, LSTM_CLASSES = 4;
const int LSTM_ENT_X = {ie["x"]}, LSTM_ENT_H = {ie["h"]}, LSTM_ENT_C = {ie["c"]};
const int LSTM_SAI_H = {isai["h"]}, LSTM_SAI_C = {isai["c"]}, LSTM_SAI_P = {isai["p"]};

// Normalização min-max dos atributos (notebook 02): temperatura, umidade, pressão - média da janela
const float FAIXA_TEMP[2] = {{{num(FX[0][0], "float")}, {num(FX[0][1], "float")}}};
const float FAIXA_UMID[2] = {{{num(FX[1][0], "float")}, {num(FX[1][1], "float")}}};
const float FAIXA_DPRESS[2] = {{{num(FX[2][0], "float")}, {num(FX[2][1], "float")}}};

// Arena mínima medida no TFLite Micro atual (o firmware reserva uma folga sobre estes valores)
const unsigned int ARENA_MINIMA_CNN = {arena_cnn}, ARENA_MINIMA_LSTM = {arena_lstm};
'''

hann = np.load(f"{MIMII}/hann.npy").astype(np.float32)
banco = np.load(f"{MIMII}/banco_mel.npy").astype(np.float32)
ini, tam, desl, pesos = [], [], [], []
for linha in banco:
    nz = np.nonzero(linha)[0]
    a, b = (int(nz[0]), int(nz[-1]) + 1) if len(nz) else (0, 0)
    ini.append(a); tam.append(b - a); desl.append(len(pesos)); pesos.extend(linha[a:b])
k = np.arange(128); kk = np.arange(256)
rev = np.array([int(f"{i:08b}"[::-1], 2) for i in range(256)])
arquivos["tabelas_mel.h"] = (CAB + "// Janela de Hann periódica (400), banco mel HTK esparso (40 filtros sobre 257 bins)\n"
    + vetor("float", "HANN", hann, 8) + vetor("uint16_t", "MEL_INICIO", ini, 20) + vetor("uint16_t", "MEL_TAM", tam, 20)
    + vetor("uint16_t", "MEL_DESLOC", desl, 20) + vetor("float", "MEL_PESOS", pesos, 8)
    + "\n// FFT real de 512 pontos = FFT complexa de 256 pontos + pós-processamento\n"
    + vetor("float", "FFT_COS", np.cos(2 * np.pi * k / 256), 8) + vetor("float", "FFT_SIN", -np.sin(2 * np.pi * k / 256), 8)
    + vetor("float", "RFFT_COS", np.cos(2 * np.pi * kk / 512), 8) + vetor("float", "RFFT_SIN", -np.sin(2 * np.pi * kk / 512), 8)
    + vetor("uint16_t", "FFT_REV", rev, 16))

# Vetores de teste: 1 janela normal + 1 anômala (paridade.npz) e uma janela real do Bristol por classe
par = np.load(f"{MIMII}/paridade.npz", allow_pickle=True)
sel = [int(np.where(par["rotulo"] == r)[0][0]) for r in (0, 1)]
maq = [re.search(r"id_\d+", str(par["membro"][i])).group(0) for i in sel]
mu = np.stack([STATS[m][0] for m in maq]); sd = np.stack([STATS[m][1] for m in maq])
lm = par["logmel"][sel].astype(np.float32)
z = (lm - mu[:, None, :]) / sd[:, None, :]
q = np.clip(np.round(z / CNN_Q_ENT[0] + CNN_Q_ENT[1]), -128, 127).astype(np.int8)
it_c, ent_c, sai_c = detalhes(cnn_tensor); saida_q = []
for i in range(2):
    it_c.set_tensor(ent_c[0]["index"], q[i][None, ..., None]); it_c.invoke()
    saida_q.append(it_c.get_tensor(sai_c[0]["index"])[0])
real = np.where(bt["sintetica"] == 0)[0] if "sintetica" in bt else np.arange(len(bt["y"]))
sel_l = [int(real[np.where(bt["y"][real] == c)[0][0]]) for c in range(4) if (bt["y"][real] == c).any()]
bruto = bt["X_bruto"][sel_l].astype(np.float32); xn = atributos(bruto); p_ref = rodar_lstm(lstm_b, xn)
arquivos["vetores_teste.h"] = (CAB + f"// Paridade: janelas {sel} de paridade.npz (rótulos 0 e 1; máquinas {maq})\n"
    + "const int N_TESTE_AUDIO = 2;\n" + vetor("int16_t", "TESTE_AUDIO", par["audio_int16"][sel], 16)
    + vetor("float", "TESTE_LOGMEL", lm, 8) + vetor("float", "TESTE_BASE_MEDIA", mu, 8) + vetor("float", "TESTE_BASE_DESVIO", sd, 8)
    + vetor("int8_t", "TESTE_CNN_ENTRADA", q, 20) + vetor("int8_t", "TESTE_CNN_SAIDA", np.array(saida_q), 2)
    + f"\n// LSTM: janelas reais do teste do Bristol, classes {[int(bt['y'][i]) for i in sel_l]}\n"
    + f"const int N_TESTE_LSTM = {len(sel_l)};\n" + vetor("int8_t", "TESTE_LSTM_CLASSE", bt["y"][sel_l], 4)
    + vetor("float", "TESTE_LSTM_BRUTO", bruto, 9) + vetor("float", "TESTE_LSTM_ATRIB", xn, 9) + vetor("float", "TESTE_LSTM_PROB", p_ref, 4))

with zipfile.ZipFile(f"{SAIDA}/firmware_headers.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for nome, txt in arquivos.items():
        open(f"{SAIDA}/{nome}", "w", encoding="utf-8").write(txt); zf.writestr(nome, txt)
for nome in arquivos:
    print(f"{nome:24s} {os.path.getsize(f'{SAIDA}/{nome}') / 1024:7.1f} KB (texto)")
print("CNN esperada (int8) nas janelas de teste:", [list(map(int, s)) for s in saida_q])

In [ ]:
# 6. Resumo para o diário e para o texto
print(f"CNN int8 (densa por tensor): {len(cnn_tensor)/1024:.1f} KB | LSTM float32: {len(lstm_b)/1024:.1f} KB | "
      f"total embarcado: {(len(cnn_tensor) + len(lstm_b))/1024:.1f} KB")
print("Operações:", OPS)
if ARENA["cnn"]:
    print(f"Arena mínima (TFLM atual): CNN {ARENA['cnn']/1024:.1f} KB, LSTM {ARENA['lstm']/1024:.1f} KB")
print("\nCopie os arquivos de", SAIDA, "(ou o firmware_headers.zip) para as pastas dos sketches:")
print("  Teste_Inferencia/              -> todos os .h")
print("  Multimodais-de-Monitorizacao/  -> todos, exceto vetores_teste.h")

## Resultado

No Drive, em `dissertacao_tinyml/firmware/`: os seis `.h` e o `firmware_headers.zip`. Em `modelos/cnn_v2/`: `cnn_int8_densa_por_tensor.tflite` e a comparação `float32_vs_int8_densa_por_tensor.csv` (atualiza as Tabelas 10 e 11, se os valores mudarem).

**Próximo passo:** gravar o sketch `Teste_Inferencia` na placa (paridade, memória e latência — Tabela 12).